In [12]:
"""
week3_task1.py
====================
Incremental update pipeline for the municipality data platform.

This script reads the three update files produced by generate_updates.py
and merges them into the existing Delta Lake tables without rebuilding them.

Schemas are read from dataset_config.json (the same config used in Week 1),
extended with the new columns introduced by each update file.

Weather     : MERGE on PK
Air Quality : MERGE on PK
Taxi Trips  : anti join and append

  You MERGE when you have a primary key to tell the database exactly how to identify a duplicate.
  You filter-then-APPEND when you lack a primary key and have to resort to comparing the entire row to deduplicate.

"""

import time
import re
import json
import os  # Added for state file check

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    FloatType, DoubleType, BooleanType, TimestampType,
)
from delta.tables import DeltaTable


spark = (
    SparkSession.builder
    .appName("IncrementalPipeline")
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Allow Delta to automatically add new columns when merging
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


# Load dataset_config.json  (same file used in ingestion.py)
with open("json_files/dataset_config.json", "r") as f:
    DATASET_CONFIGS = json.load(f)

# Map config type strings -> Spark type objects
SPARK_TYPE_MAP = {
    "integer"  : IntegerType(),
    "int"      : IntegerType(),
    "long"     : LongType(),
    "bigint"   : LongType(),
    "float"    : FloatType(),
    "double"   : DoubleType(),
    "string"   : StringType(),
    "boolean"  : BooleanType(),
    "bool"     : BooleanType(),
    "timestamp": TimestampType(),
}


# Initialization Helpers (Migrated from task2)

STATE_PATH = "json_files/pipeline_state.json"
RAW_TABLES = {
    "weather": "delta/weather",
    "air_quality": "delta/air_quality",
    "taxi_trips_03": "delta/taxi_trips_03",
}

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "r") as f:
            return json.load(f)
    return {"tables": {}}

def save_state(state):
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)

def table_fingerprint(path):
    dt = DeltaTable.forPath(spark, path)
    version = dt.history(1).select("version").collect()[0]["version"]
    columns = spark.read.format("delta").load(path).columns
    return version, columns

def initialize_state_if_needed():
    state = load_state()
    if len(state["tables"]) == 0:
        print("Initializing pipeline state for change detection (bootstrap)...")
        for name, path in RAW_TABLES.items():
            try:
                version, columns = table_fingerprint(path)
                state["tables"][name] = {"version": version, "columns": columns}
            except Exception as e:
                print(f"Skipping {name} baseline: {e}")
        save_state(state)
        print("Baseline saved. Proceeding with incremental updates.\n")


def to_snake_case(name):
    s  = name.strip()
    s  = re.sub(r'[\s\-]+', '_', s)
    s1 = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
    s3 = re.sub(r'_+', '_', s2)
    return s3.lower()

def standardize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df


# Helper: build a Spark StructType from the config schema dict,
#         optionally appending extra columns (for schema evolution).
#
# extra_cols is a dict like {"humidity": DoubleType()} for new columns.
# All fields are marked nullable=True (consistent with how CSVs are read).
def build_schema(config_name, extra_cols=None):
    config = DATASET_CONFIGS[config_name]
    fields = []
    for col_name, type_str in config["schema"].items():
        snake = to_snake_case(col_name)
        spark_type = SPARK_TYPE_MAP.get(type_str.lower(), StringType())
        fields.append(StructField(snake, spark_type, nullable=True))

    # Append new columns introduced by this update (schema evolution)
    if extra_cols:
        for col_name, spark_type in extra_cols.items():
            fields.append(StructField(col_name, spark_type, nullable=True))

    return StructType(fields)


# Helper: align DataFrame schema with existing Delta table schema
def normalize_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            c = field.name
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))
    return df

def assemble_timestamp_from_parts(df, config):
    if not config:
        return df
    year   = config["year_col"]
    month  = config["month_col"]
    day    = config["day_col"]
    hour   = config.get("hour_col")
    minute = config.get("minute_col")
    second = config.get("second_col")
    out    = config["output_col"]

    time_part = F.concat(
        F.lpad(F.col(year).cast("string"), 4, "0"), F.lit("-"),
        F.lpad(F.col(month).cast("string"), 2, "0"), F.lit("-"),
        F.lpad(F.col(day).cast("string"), 2, "0"), F.lit(" "),
        F.lpad(F.col(hour).cast("string"), 2, "0") if hour else F.lit("00"), F.lit(":"),
        F.lpad(F.col(minute).cast("string"), 2, "0") if minute else F.lit("00"), F.lit(":"),
        F.lpad(F.col(second).cast("string"), 2, "0") if second else F.lit("00"),
    )
    return df.withColumn(out, F.to_timestamp(time_part, "yyyy-MM-dd HH:mm:ss"))

def normalize_date_time_pairs(df, pairs):
    if not pairs:
        return df
    for pair in pairs:
        date_col   = to_snake_case(pair["date_col"])
        time_col   = to_snake_case(pair["time_col"])
        output_col = to_snake_case(pair["output_col"])
        fmt        = pair["format"]
        datetime_str = F.concat(F.col(date_col), F.lit(" "), F.col(time_col))
        df = df.withColumn(output_col, F.to_timestamp(datetime_str, fmt))
    return df

def align_to_target_table(df, delta_path, new_cols_schema=None):
    """
    Aligns incoming df column types to match the target Delta table's existing schema.
    Converts timestamps, dates, booleans, and numerics so Delta won't fail to merge fields.
    """
    target_schema = spark.read.format("delta").load(delta_path).schema

    truthy = {"y", "yes", "true", "1"}
    falsy  = {"n", "no",  "false", "0"}

    for field in target_schema.fields:
        c = field.name
        if c in df.columns:
            target_type = field.dataType
            if isinstance(target_type, BooleanType):
                lowered = F.lower(F.trim(F.col(c).cast("string")))
                df = df.withColumn(
                    c,
                    F.when(lowered.isin(*truthy), F.lit(True))
                    .when(lowered.isin(*falsy),  F.lit(False))
                    .otherwise(None)
                    .cast(BooleanType())
                )
            elif isinstance(target_type, TimestampType):
                df = df.withColumn(c, F.to_timestamp(F.col(c)))
            elif str(target_type).lower().startswith("date"):
                df = df.withColumn(c, F.to_date(F.col(c)))
            else:
                df = df.withColumn(c, F.col(c).cast(target_type))

    if new_cols_schema:
        for col_name, col_type in new_cols_schema.items():
            if col_name in df.columns:
                df = df.withColumn(col_name, F.col(col_name).cast(col_type))

    return df


# Helper: build a MERGE condition string from a list of key columns
def build_merge_condition(keys):
    parts = [f"existing.{k} = new.{k}" for k in keys]
    return " AND ".join(parts)



# 1.  WEATHER  –  MERGE on (year, month, day, hour)
def update_weather():
    print("=" * 60)
    print("Updating weather table ...")
    t0 = time.time()

    delta_path  = "delta/weather"
    update_path = "datasets/updates/weather_update.csv"

    #  Load update CSV and apply standardizations 
    update_df = spark.read.option("header", "true").csv(update_path)
    update_df = standardize_columns(update_df)
    update_df = normalize_string_columns(update_df)
    update_df = assemble_timestamp_from_parts(update_df, DATASET_CONFIGS["weather"].get("assembled_timestamp"))

    # Dynamically detect new columns 
    existing_df = spark.read.format("delta").load(delta_path)
    existing_cols = set(existing_df.columns)
    new_cols = [c for c in update_df.columns if c not in existing_cols]
    
    schema_change_str = f"added column(s): {', '.join(new_cols)}" if new_cols else "none"

    # Align to target table (no hardcoded new_cols_schema needed)
    update_df = align_to_target_table(update_df, delta_path)

    n_update = update_df.count()

    #  Count how many update rows are already in the existing table
    pk = ["year", "month", "day", "hour"]
    matched_count = (
        update_df.alias("u")
        .join(existing_df.select(pk).alias("e"),
              on=[F.col(f"u.{k}") == F.col(f"e.{k}") for k in pk],
              how="inner")
        .count()
    )

    #  Open the existing Delta table and run MERGE
    delta_table = DeltaTable.forPath(spark, delta_path)

    (
        delta_table.alias("existing")
        .merge(
            update_df.alias("new"),
            build_merge_condition(pk)
        )
        .whenNotMatchedInsertAll()   # only inserts; never updates existing rows
        .execute()
    )

    n_inserted = n_update - matched_count
    elapsed    = round(time.time() - t0, 2)

    print(f"  Rows in update file  : {n_update:,}")
    print(f"  Duplicates skipped   : {matched_count:,}")
    print(f"  New rows inserted    : {n_inserted:,}")
    print(f"  Schema change        : {schema_change_str}")
    print(f"  Elapsed              : {elapsed} s")
    print()

    return {
        "dataset"       : "weather",
        "rows_in_update": n_update,
        "duplicates"    : matched_count,
        "inserted"      : n_inserted,
        "schema_change" : schema_change_str,
        "elapsed_s"     : elapsed,
    }


# 2.  AIR QUALITY  –  MERGE on 7-column composite PK
def update_air_quality():
    print("=" * 60)
    print("Updating air quality table ...")
    t0 = time.time()

    delta_path  = "delta/air_quality"
    update_path = "datasets/updates/air_quality_update.csv"

    #  Load update CSV and apply standardizations 
    update_df = spark.read.option("header", "true").csv(update_path)
    update_df = standardize_columns(update_df)
    update_df = normalize_string_columns(update_df)
    update_df = normalize_date_time_pairs(update_df, DATASET_CONFIGS["air_quality"].get("date_time_pairs", []))
    
    #  Dynamically detect new columns 
    existing_df   = spark.read.format("delta").load(delta_path)
    existing_cols = set(existing_df.columns)
    new_cols = [c for c in update_df.columns if c not in existing_cols]
    
    schema_change_str = f"added column(s): {', '.join(new_cols)}" if new_cols else "none"

    # Align to target table (no hardcoded new_cols_schema needed)
    update_df = align_to_target_table(update_df, delta_path)

    n_update = update_df.count()

    # Primary key columns (snake_case, from dataset_config.json)
    pk = ["state_code", "county_code", "site_num", "parameter_code",
          "poc", "date_local", "time_local"]

    #  Count duplicates via inner join on the 7-column PK 
    existing_keys = existing_df.select(pk).dropDuplicates()
    matched_count = update_df.join(existing_keys, on=pk, how="left_semi").count()

    #  MERGE 
    delta_table = DeltaTable.forPath(spark, delta_path)

    (
        delta_table.alias("existing")
        .merge(
            update_df.alias("new"),
            build_merge_condition(pk)
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    n_inserted = n_update - matched_count
    elapsed    = round(time.time() - t0, 2)

    print(f"  Rows in update file  : {n_update:,}")
    print(f"  Duplicates skipped   : {matched_count:,}")
    print(f"  New rows inserted    : {n_inserted:,}")
    print(f"  Schema change        : {schema_change_str}")
    print(f"  Elapsed              : {elapsed} s")
    print()

    return {
        "dataset"       : "air_quality",
        "rows_in_update": n_update,
        "duplicates"    : matched_count,
        "inserted"      : n_inserted,
        "schema_change" : schema_change_str,
        "elapsed_s"     : elapsed,
    }



# 3.  TAXI TRIPS  –  No natural PK: remove duplicates + append
def update_taxi_trips():
    print("=" * 60)
    print("Updating taxi trips table ...")
    t0 = time.time()

    update_path = "datasets/updates/yellow_tripdata_update.parquet"

    update_df = spark.read.parquet(update_path)
    update_df = standardize_columns(update_df)
    update_df = align_to_target_table(update_df, "delta/taxi_trips_03")

    n_update = update_df.count()
    print(f"  Rows in update file  : {n_update:,}")

    existing = (
        spark.read.format("delta").load("delta/taxi_trips_01")
        .unionByName(spark.read.format("delta").load("delta/taxi_trips_02"))
        .unionByName(spark.read.format("delta").load("delta/taxi_trips_03"))
    )

    cols = update_df.columns
    existing_sel = existing.select(cols)

    join_cond = [update_df[c].eqNullSafe(existing_sel[c]) for c in cols]

    unique_new = (
        update_df
        .join(existing_sel, on=join_cond, how="left_anti")
        .dropDuplicates()
    )

    n_inserted  = unique_new.count()
    n_duplicate = n_update - n_inserted
    print(f"  Duplicates detected  : {n_duplicate:,}")
    print(f"  New rows to insert   : {n_inserted:,}")

    (
        unique_new.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save("delta/taxi_trips_03")
    )

    elapsed = round(time.time() - t0, 2)
    print(f"  Schema change        : none (taxi trips update has same schema)")
    print(f"  Elapsed              : {elapsed} s")
    print()

    return {
        "dataset"       : "taxi_trips",
        "rows_in_update": n_update,
        "duplicates"    : n_duplicate,
        "inserted"      : n_inserted,
        "schema_change" : "none",
        "elapsed_s"     : elapsed,
    }


# Verify: show Delta table history to confirm new versions were created
def show_delta_history(delta_path, n=3):
    dt = DeltaTable.forPath(spark, delta_path)
    dt.history(n).select("version", "timestamp", "operation", "operationMetrics").show(
        n, truncate=False
    )



# MAIN

if __name__ == "__main__":
    # Create the pre-update baseline if it doesn't exist
    initialize_state_if_needed()

    results = []

    results.append(update_weather())
    results.append(update_air_quality())
    results.append(update_taxi_trips())

    # Final summary
    print("=" * 60)
    print("INCREMENTAL PIPELINE SUMMARY")
    print("=" * 60)
    print()
    for r in results:
        print(f"Dataset         : {r['dataset']}")
        print(f"  Rows in update: {r['rows_in_update']:,}")
        print(f"  Duplicates    : {r['duplicates']:,}  (skipped)")
        print(f"  Inserted      : {r['inserted']:,}")
        print(f"  Schema change : {r['schema_change']}")
        print(f"  Time          : {r['elapsed_s']} s")
        print()


    #  Confirm new columns are present 
    print("\nSchema of delta/weather (should include 'humidity'):")
    spark.read.format("delta").load("delta/weather").printSchema()

    print("\nSchema of delta/air_quality (should include 'aqi'):")
    spark.read.format("delta").load("delta/air_quality").printSchema()

    spark.stop()

26/09/23 20:17:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/23 20:17:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/23 20:17:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Updating weather table ...


  Rows in update file  : 240
  Duplicates skipped   : 0
  New rows inserted    : 240
  Schema change        : added column(s): humidity
  Elapsed              : 7.62 s

Updating air quality table ...


  Rows in update file  : 1,440
  Duplicates skipped   : 0
  New rows inserted    : 1,440
  Schema change        : added column(s): aqi
  Elapsed              : 22.28 s

Updating taxi trips table ...
  Rows in update file  : 304,524


  Duplicates detected  : 53,740
  New rows to insert   : 250,784


  Schema change        : none (taxi trips update has same schema)
  Elapsed              : 28.73 s

INCREMENTAL PIPELINE SUMMARY

Dataset         : weather
  Rows in update: 240
  Duplicates    : 0  (skipped)
  Inserted      : 240
  Schema change : added column(s): humidity
  Time          : 7.62 s

Dataset         : air_quality
  Rows in update: 1,440
  Duplicates    : 0  (skipped)
  Inserted      : 1,440
  Schema change : added column(s): aqi
  Time          : 22.28 s

Dataset         : taxi_trips
  Rows in update: 304,524
  Duplicates    : 53,740  (skipped)
  Inserted      : 250,784
  Schema change : none
  Time          : 28.73 s


Schema of delta/weather (should include 'humidity'):
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- temp: double (nullable = true)
 |-- temp_source: string (nullable = true)
 |-- rhum: double (nullable = true)
 |-- rhum_source: string (nullable 